In [1]:
import os
import sys

os.chdir("..")
sys.path.append(os.getcwd())

import pandas as pd
import duckdb
from pathlib import Path
from config import RESEARCH_ROOT, NSE_DB_PATH

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

import numpy as np
from scipy import stats

In [ ]:
RESEARCH_DB_PATH = RESEARCH_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)")

In [7]:
feature_groups = {
    'price_vol': ['nifty_close', 'vix_close', 'underlying_daily_vol', 'futures_daily_vol', 'applicable_daily_vol'],
    'options_derived': ['pcr', 'max_pain_dist_pct', 'basis', 'cost_of_carry', 'fut_chng_oi_pct'],
    'breadth': ['advances', 'declines', 'adv_decl_ratio', 'price_band_hits', 'traded_value_cr', 'num_trades', 'market_cap_cr'],
    'fii_dii_positioning': ['fii_fut_net_pct', 'client_fut_net_pct', 'fii_pe_net_pct', 'fii_net_flow_cr',
                            'dii_fut_net_pct', 'pro_fut_net_pct', 'fii_stk_fut_net_pct', 'client_stk_fut_net_pct'],
    'options_skew': ['fii_opt_skew_pct', 'client_opt_skew_pct', 'dii_opt_skew_pct', 'pro_opt_skew_pct',
                      'fii_vol_opt_skew_pct', 'client_vol_opt_skew_pct'],
    'vol_futures_flow': ['fii_vol_fut_net_pct', 'client_vol_fut_net_pct', 'pro_vol_fut_net_pct'],
    'divergence_stats': ['fii_client_divergence', 'fii_stats_fut_net_pct', 'fii_stats_oi_net_cr'],
    'engineered_deltas': ['vix_chg_5d', 'pcr_chg_5d', 'basis_chg_5d', 'max_pain_dist_chg_5d', 
                           'vix_realized_spread', 'fii_fut_net_chg_5d'],
    'targets': ['fwd_ret_1d', 'fwd_ret_5d', 'fwd_ret_20d', 'up_1d']
}

emission_raw = ['vix_close', 'underlying_daily_vol', 'price_band_hits', 
                 'fii_stats_fut_net_pct', 'client_opt_skew_pct', 'max_pain_dist_pct', 'basis_chg_5d']

In [8]:
df_work = pd.read_csv(RESEARCH_ROOT / '8_state_data.csv', index_col=0, parse_dates=True)
print(df_work.shape)
print(df_work['hmm_state_v2'].notna().sum(), "rows with HMM state")

model_df = df_work.dropna(subset=['hmm_state_v2']).copy()
model_df['hmm_state_v2'] = model_df['hmm_state_v2'].astype(int)

target = 'fwd_ret_20d'
feature_cols_model = [c for c in [c for g,cols in feature_groups.items() if g!='targets' for c in cols] 
                       if c not in emission_raw]  # exclude HMM's own inputs

model_df = model_df.dropna(subset=[target] + feature_cols_model)
print(f"\nUsable rows: {len(model_df)}")
print(f"Date range: {model_df.index.min()} to {model_df.index.max()}")
print(f"Feature count: {len(feature_cols_model)}")

(2616, 48)
2552 rows with HMM state

Usable rows: 2411
Date range: 2016-04-06 00:00:00 to 2026-06-29 00:00:00
Feature count: 36


In [10]:
expiries = con.execute("""
    SELECT DISTINCT expiry
    FROM nse.instruments
    WHERE ticker = 'NIFTY'
      AND expiry IS NOT NULL
    ORDER BY expiry
""").fetchdf()

expiries['expiry'] = pd.to_datetime(expiries['expiry'])

# filter to within our actual data range — drops the far-dated 2029 etc futures listings
expiries_in_range = expiries[(expiries['expiry'] >= df_work.index.min()) & 
                               (expiries['expiry'] <= df_work.index.max())]
print(f"Total distinct expiries in range: {len(expiries_in_range)}")

# monthly = last expiry of each calendar month (standard convention: last Thursday, 
# even after weekly expiries started, the LAST expiry of the month is still "the monthly")
expiries_in_range = expiries_in_range.copy()
expiries_in_range['year_month'] = expiries_in_range['expiry'].dt.to_period('M')
monthly_expiries = expiries_in_range.groupby('year_month')['expiry'].max().reset_index(drop=True)

print(f"\nMonthly expiries identified: {len(monthly_expiries)}")
print(monthly_expiries.head(10))
print(monthly_expiries.tail(10))

# compare to our cost_of_carry NaN proxy
coc_nan_dates = set(df_work[df_work['cost_of_carry'].isna()].index.date)
monthly_expiry_dates = set(monthly_expiries.dt.date)

print(f"\ncost_of_carry NaN count: {len(coc_nan_dates)}")
print(f"True monthly expiry count: {len(monthly_expiry_dates)}")
print(f"In both: {len(coc_nan_dates & monthly_expiry_dates)}")
print(f"In coc_nan but NOT true monthly expiry: {sorted(coc_nan_dates - monthly_expiry_dates)[:10]}")
print(f"In true monthly expiry but NOT coc_nan: {sorted(monthly_expiry_dates - coc_nan_dates)[:10]}")

Total distinct expiries in range: 434

Monthly expiries identified: 127
0   2016-01-28
1   2016-02-25
2   2016-03-31
3   2016-04-28
4   2016-05-26
5   2016-06-30
6   2016-07-28
7   2016-08-25
8   2016-09-29
9   2016-10-27
Name: expiry, dtype: datetime64[us]
117   2025-10-28
118   2025-11-25
119   2025-12-30
120   2026-01-27
121   2026-02-24
122   2026-03-31
123   2026-04-28
124   2026-05-26
125   2026-06-30
126   2026-07-21
Name: expiry, dtype: datetime64[us]

cost_of_carry NaN count: 126
True monthly expiry count: 127
In both: 121
In coc_nan but NOT true monthly expiry: [datetime.date(2018, 3, 28), datetime.date(2023, 3, 29), datetime.date(2023, 6, 28), datetime.date(2025, 4, 24), datetime.date(2026, 3, 30)]
In true monthly expiry but NOT coc_nan: [datetime.date(2018, 3, 29), datetime.date(2023, 3, 30), datetime.date(2023, 6, 29), datetime.date(2025, 4, 30), datetime.date(2026, 3, 31), datetime.date(2026, 7, 21)]


In [11]:
# use only expiries that have actually occurred within our data
valid_monthly_expiries = monthly_expiries[monthly_expiries <= df_work.index.max()].reset_index(drop=True)
print(f"Usable monthly expiry boundaries: {len(valid_monthly_expiries)}")

# walk-forward: train up to expiry[i], test = the period between expiry[i] and expiry[i+1]
# start after we have a reasonable minimum training history (e.g. first 24 expiries ~ 2 years)
min_train_expiries = 24
embargo_days = 20  # same overlap-purge logic as before, for fwd_ret_20d

folds = []
for i in range(min_train_expiries, len(valid_monthly_expiries) - 1):
    train_end = valid_monthly_expiries.iloc[i]
    test_start = valid_monthly_expiries.iloc[i]
    test_end = valid_monthly_expiries.iloc[i + 1]
    
    train_mask = model_df.index <= (train_end - pd.Timedelta(days=embargo_days))
    test_mask = (model_df.index > test_start) & (model_df.index <= test_end)
    
    train_idx = model_df.index[train_mask]
    test_idx = model_df.index[test_mask]
    
    if len(train_idx) < 100 or len(test_idx) < 5:
        continue
    folds.append({'train_idx': train_idx, 'test_idx': test_idx, 
                   'train_end': train_end, 'test_start': test_start, 'test_end': test_end})

print(f"\nTotal folds: {len(folds)}")
for f in folds[:3] + folds[-3:]:
    print(f"train ends {f['train_end'].date()} | test: {f['test_start'].date()} to {f['test_end'].date()} "
          f"| train n={len(f['train_idx'])}, test n={len(f['test_idx'])}")

Usable monthly expiry boundaries: 127

Total folds: 101
train ends 2018-01-25 | test: 2018-01-25 to 2018-02-22 | train n=415, test n=17
train ends 2018-02-22 | test: 2018-02-22 to 2018-03-29 | train n=433, test n=22
train ends 2018-03-29 | test: 2018-03-29 to 2018-04-26 | train n=455, test n=18
train ends 2026-03-31 | test: 2026-03-31 to 2026-04-28 | train n=2343, test n=17
train ends 2026-04-28 | test: 2026-04-28 to 2026-05-26 | train n=2359, test n=18
train ends 2026-05-26 | test: 2026-05-26 to 2026-06-30 | train n=2376, test n=22


In [12]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

interaction_feats = ['futures_daily_vol', 'fii_stk_fut_net_pct', 'dii_opt_skew_pct']

def build_interaction_matrix(df, base_feats, interaction_feats, n_states=8):
    X = df[base_feats].copy()
    state_dummies = pd.get_dummies(df['hmm_state_v2'], prefix='state')
    for feat in interaction_feats:
        for s in range(n_states):
            col = f'{feat}_x_state{s}'
            X[col] = df[feat] * (df['hmm_state_v2'] == s).astype(int)
    X = pd.concat([X, state_dummies], axis=1)
    return X

base_feats = [c for c in feature_cols_model if c not in interaction_feats]

fold_results_linear = []
for i, f in enumerate(folds):
    train = model_df.loc[f['train_idx']]
    test = model_df.loc[f['test_idx']]
    
    X_train = build_interaction_matrix(train, base_feats, interaction_feats)
    X_test = build_interaction_matrix(test, base_feats, interaction_feats)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)  # handle missing states in test fold
    
    y_train, y_test = train[target], test[target]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    model = Ridge(alpha=10.0)
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    
    ic = np.corrcoef(preds, y_test)[0,1] if len(y_test) > 2 else np.nan
    mse = np.mean((preds - y_test)**2)
    fold_results_linear.append({'fold': i, 'test_start': f['test_start'], 'ic': ic, 'mse': mse, 'n': len(y_test)})

linear_results_df = pd.DataFrame(fold_results_linear)
print(linear_results_df[['ic','mse']].describe())
print(f"\nMean IC across folds: {linear_results_df['ic'].mean():.4f}")
print(f"IC t-stat (mean/se): {linear_results_df['ic'].mean() / (linear_results_df['ic'].std()/np.sqrt(len(linear_results_df))):.3f}")

               ic         mse
count  101.000000  101.000000
mean     0.119844   53.632220
std      0.449341  102.192270
min     -0.881169    1.479226
25%     -0.187279    8.235964
50%      0.140672   20.381224
75%      0.454570   55.413717
max      0.987124  693.013510

Mean IC across folds: 0.1198
IC t-stat (mean/se): 2.680


In [ ]:
linear_results_df['test_start'] = pd.to_datetime(linear_results_df['test_start'])
linear_results_df = linear_results_df.sort_values('test_start')

# 1. IC vs training size - is instability just an early-fold small-sample problem?
train_sizes = [len(f['train_idx']) for f in folds]
linear_results_df['train_size'] = train_sizes
print(linear_results_df[['train_size','ic','mse']].corr())

# 2. worst and best folds - what dates/regimes were they?
print("\nWorst 5 folds (by IC):")
print(linear_results_df.nsmallest(5, 'ic')[['test_start','ic','mse','train_size']])
print("\nBest 5 folds (by IC):")
print(linear_results_df.nlargest(5, 'ic')[['test_start','ic','mse','train_size']])

# 3. IC over time - plot
fig, ax = plt.subplots(figsize=(14,4))
ax.plot(linear_results_df['test_start'], linear_results_df['ic'], marker='o', markersize=3)
ax.axhline(0, color='gray', linestyle='--')
ax.set_title('Fold IC over time')
plt.tight_layout()
plt.savefig('linear_ic_over_time.png', dpi=100)
plt.show()

# 4. exclude first 20 folds (smallest training windows) and recompute
stable_folds = linear_results_df.iloc[20:]
print(f"\nExcluding first 20 folds (small training size):")
print(f"Mean IC: {stable_folds['ic'].mean():.4f}, t-stat: {stable_folds['ic'].mean()/(stable_folds['ic'].std()/np.sqrt(len(stable_folds))):.3f}")